## Step 3: Building a Prediction Model

Importations des Modules

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

Importe le fichier csv final et supprime les lignes du tableau qui contiennent des données manquantes dans les colonnes cités

In [ ]:
df = pd.read_csv("cleaned_dataset.csv")
df = df.dropna(subset=["Gare de départ", "Gare d'arrivée", "Mois", "Service", "Retard moyen de tous les trains à l'arrivée"])

On définit le LabelEncoder dans une variable. On filtre les retards moyens des trains à l'arrivée supérieurs ou égaux à 0.

In [ ]:
le = LabelEncoder()
df = df[df["Retard moyen de tous les trains à l'arrivée"] >= 0]

Ici, on remplace toutes les colonnes par des chiffres, chaque gare se voit donc attribuer un chiffre.  <br><br>  Cela permet à l'IA de mieux traiter les données ou de s'y retrouver.

In [ ]:
encoders = {}

le_depart = LabelEncoder()
le_arrivee = LabelEncoder()
le_service = LabelEncoder()

df["Gare de départ"] = le_depart.fit_transform(df["Gare de départ"].astype(str))
df["Gare d'arrivée"] = le_arrivee.fit_transform(df["Gare d'arrivée"].astype(str))
df["Service"] = le_service.fit_transform(df["Service"].astype(str))

encoders["depart"] = le_depart
encoders["arrivee"] = le_arrivee
encoders["service"] = le_service

df["Nombre de circulations prévues"] = le.fit_transform(df["Nombre de circulations prévues"].astype(str))
df["Nombre de trains annulés"] = le.fit_transform(df["Nombre de trains annulés"].astype(str))

Nettoyage de la durée : on enlève " min", on remplace la virgule par un point, conversion en nombre.

In [ ]:
df["Durée moyenne du trajet"] = (df["Durée moyenne du trajet"].astype(str).str.replace(" min", "").str.replace(",", "."))

df["Durée moyenne du trajet"] = pd.to_numeric(df["Durée moyenne du trajet"], errors='coerce')

df["Durée moyenne du trajet"] = df["Durée moyenne du trajet"].fillna(df["Durée moyenne du trajet"].mean())

Cette syntaxe définit les variables neccésaire à l'apprentissage :  <br><br>
X = Les caractéristique (features) les données qui influencent la ponctualité.  
y = La cible (target) la valeur que nous cherchons a prédire.

In [ ]:
X = df[["Gare de départ", "Gare d'arrivée", "Mois", "Service", "Durée moyenne du trajet", "Annee", "Nombre de circulations prévues"]]
y = df["Retard moyen de tous les trains à l'arrivée"]

On divisie les données pour entrainer un modèle Random Forest sur 80% du dataset et on évalue sa capacité à prédire les retards sur les 20 % restants.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modele = RandomForestRegressor(n_estimators=100).fit(X=X_train, y=y_train)

y_pred = modele.predict(X_test)


On définit les variables MAE, RMSE et R2 grâce aux bibliothèques que l'on a importées.

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = root_mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

Affichage des résultats des données  :  <br><br>

Mean Absolute Error (MAE)  

Root Mean Squared Error (RMSE)  

Coefficient de détermination (R2)  



In [ ]:
print(f"MAE : {mae:.2f}")
print(f"RMSE : {rmse:.2f}")
print(f"R2 : {r2:.2f}")

MAE : 2.03
RMSE : 3.32
R2 : 0.44


Sauvegarde de mon intelligence artificielle pour pouvoir l'utiliser dans le dashboard pour passer l'étape 4. 

In [ ]:
import joblib

joblib.dump(modele, "model.joblib")

joblib.dump(X_train.columns.tolist(), "model_columns.joblib")

joblib.dump(encoders, "encoders.joblib")


['encoders.joblib']